In [0]:

from pyspark.sql.functions import col, lower, trim, count, when, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

#load data 
df_bronze_match_1 = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2015_2018")
df_bronze_match_2 = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2018_2026")
#checking schema 
df_bronze_match_1.printSchema()
df_bronze_match_1.show(1000)

df_bronze_match_2.show(1000)


In [0]:

#removing all null values 
df_bronze_match_1 = df_bronze_match_1.na.drop(subset = ["HomeTeam", "AwayTeam", "Season", "Round", "HomeScore", "AwayScore"])
df_bronze_match_2 = df_bronze_match_2.na.drop(subset = ["HomeTeam", "AwayTeam", "Season", "Round", "HomeScore", "AwayScore"])

#merge both dataframes
df_merged = df_bronze_match_1.unionByName(df_bronze_match_2, allowMissingColumns=True)

#check for any missing fields in the merged dataset
df_merged.select([
    count(when(col(c).isNull(), c)).alias(c) for c in ["HomeTeam", "AwayTeam", "Season", "Round", "HomeScore", "AwayScore"]
])

#check for duplicated 
duplicates_count = df_merged.count() - df_merged.dropDuplicates().count()
print(f"Number of duplicates: {duplicates_count}")

#drop duplicates 
df_merged = df_merged.dropDuplicates()

#check all duplicates are removed
duplicates_count = df_merged.count() - df_merged.dropDuplicates().count()
print(f"Number of duplicates: {duplicates_count}")

#check for invalid values
df_merged.filter((col("HomeScore") < 0) | (col("AwayScore") < 0)).show()


In [0]:
#standardise team names

#create map to make teams consistent 
team_map = {
    "bath rugby": "bath",
    "bath rugby club": "bath",
    "bristol bears": "bristol",
    "bristol rugby club": "bristol",
    "bears": "bristol",
    "chiefs": "exeter-chiefs",
    "exeter-cheifs": "exeter-chiefs",
    "gloucester rugby": "gloucester",
    "gloucester rugby club": "gloucester",
    "cherries": "gloucester",
    "harlequins rugby": "harlequins",
    "harlequins rugby club": "harlequins",
    "leicester tigers": "leicester",
    "leicester tigers rugby club": "leicester",
    "leicester": "leicester",
    "london irish": "london-irish",
    "london irish rugby club": "london-irish",
    "northampton saints": "northampton",
    "northampton saints rugby club": "northampton",
    "northampton": "northampton",
    "saints": "northampton",
    "sale sharks": "sale",
    "sale rugby club": "sale",
    "wasps rugby": "wasps",
    "wasps rugby club": "wasps",
    "saracens rugby": "saracens",
    "saracens rugby club": "saracens",
    "sarries": "saracens",
    "worcester warriors": "worcester",
    "worcester rugby club": "worcester",
    "worcester": "worcester",
    "newcastle falcons": "newcastle",
    "newcastle rugby club": "newcastle",
    "falcons": "newcastle",
    "newcastle red bulls": "newcastke",
    "newcastle red bulls rugby club": "newcastle"
}

#set all team names to lower case and remove leading and trailing whitespaces
for column in ["HomeTeam", "AwayTeam"]:
    df_merged = df_merged.withColumn(column, trim(lower(col(column))))

df_merged = df_merged.replace(team_map, subset = ["HomeTeam", "AwayTeam"])


In [0]:
#add new columns by calculating outcomes

#creates a new column for the match result
df_merged = df_merged.withColumn("Result", 
    when(col("HomeScore") > col("AwayScore"), "Home Win")
    .when(col("HomeScore") < col("AwayScore"), "Away Win")
    .otherwise("Draw")
)

#creates a new column for the points difference
df_merged = df_merged.withColumn(
    "MatchPointsDifference",
    (col("HomeScore") - col("AwayScore")).cast("int")
)

df_merged = df_merged.withColumn(
    "HomePointsDifference",
    (col("HomeScore") - col("AwayScore")).cast("int")
)

df_merged = df_merged.withColumn(
    "AwayPointsDifference",
    (col("AwayScore") - col("HomeScore")).cast("int")
)

In [0]:
#creates a new schema to enforce
schema = StructType([
    StructField("MatchId", IntegerType(), True),
    StructField("HomeTeam", StringType(), True),
    StructField("AwayTeam", StringType(), True),
    StructField("Season", StringType(), True),
    StructField("Round", StringType(), True),
    StructField("HomeScore", IntegerType(), True),
    StructField("AwayScore", IntegerType(), True),
    StructField("Result", StringType(), True),
    StructField("MatchPointsDifference", IntegerType(), True),
    StructField("HomePointsDifference", IntegerType(), True),
    StructField("AwayPointsDifference", IntegerType(), True),
    StructField("ingestTimestamp", TimestampType(), True),
    StructField("sourceFile", StringType(), True),
])


In [0]:
#write merged dataframe to silver schema
#rename bronze meta data columns 
df_merged = df_merged.withColumnRenamed("ingestTimestamp", "ingestTimestampBronze").withColumnRenamed("sourceFile", "sourceFileBronze")
#add silver layer meta data 
df_merged = df_merged.withColumn("lastUpload", current_timestamp()).withColumn("pipelineStage", lit("silverTransformation"))

#write to silver table
df_merged.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("rugby_data_dev.rugby_silver.match_results")

In [0]:
df_bronze_match_data = spark.read.table("rugby_data_dev.rugby_bronze.match_results_raw_2022_2026")
#df_bronze_match_data.show(100)

#remove null values 
df_bronze_match_data = df_bronze_match_data.na.drop(subset = ["HomeTeam", "AwayTeam", "Season", "Round", "HomeScore", "AwayScore"])

#check for any missing fields in the merged dataset
df_bronze_match_data.select([
    count(when(col(c).isNull(), c)).alias(c) for c in ["HomeTeam", "AwayTeam", "Season", "Round", "HomeScore", "AwayScore"]
]).show()

#check for duplicated 
duplicates_count = df_bronze_match_data.count() - df_bronze_match_data.dropDuplicates().count()
print(f"Number of duplicates: {duplicates_count}")

#remove duplicates 
df_bronze_match_data = df_bronze_match_data.dropDuplicates()

#check for any remaining duplicates
duplicates_count = df_bronze_match_data.count() - df_bronze_match_data.dropDuplicates().count()
print(f"Number of duplicates: {duplicates_count}")

#convert names 
for column in ["HomeTeam", "AwayTeam"]: 
    df_bronze_match_data = df_bronze_match_data.withColumn(column, trim(lower(col(column))))

df_bronze_match_data = df_bronze_match_data.replace(team_map, subset=["HomeTeam", "AwayTeam"])


In [0]:
#check for invalid values
#check score is not negative 
df_bronze_match_data.filter((col("HomeScore") < 0) | (col("AwayScore") < 0)).show()

#check conversions <= tries 
df_bronze_match_data.filter((col("HomeTries") < col("HomeConversions")) | (col("AwayTries") < col("AwayConversions"))).show()

# #check metres gained > post contact metres etc... 
df_bronze_match_data.filter((col("HomeMetresGained") < col("HomePostContactMetres")) | (col("AwayMetresGained") < col("AwayPostContactMetres"))).show()

#cap post contanct meters that are greater than metres gained 
df_bronze_match_data = df_bronze_match_data.withColumn("HomePostContactMetres", when(col("HomePostContactMetres") > col("HomeMetresGained"), col("HomeMetresGained")).otherwise(col("HomePostContactMetres")))
df_bronze_match_data = df_bronze_match_data.withColumn("AwayPostContactMetres", when(col("AwayPostContactMetres") > col("AwayMetresGained"), col("AwayMetresGained")).otherwise(col("AwayPostContactMetres")))

#Territory and Pocession 
df_bronze_match_data.filter((col("HomeTerritory") + col("AwayTerritory")) != 1).show()
df_bronze_match_data.filter((col("HomePossession") + col("AwayPossession")) != 1).show()

#Score check
df_bronze_match_data.filter(
    (col("HomeScore") != col("HomeTries") * 5 + col("HomeConversions") * 2 + col("HomePenaltyGoals") * 3) |
    (col("AwayScore") != col("AwayTries") * 5 + col("AwayConversions") * 2 + col("AwayPenaltyGoals") * 3)
).show()

#ruck speeds add up to 100%
df_bronze_match_data.filter((col("Home0-3sRuckSpeed") + col("Home3-6sRuckSpeed") + col("Home6s+RuckSpeed")) != 1).show()
df_bronze_match_data.filter((col("Away0-3sRuckSpeed") + col("Away3-6sRuckSpeed") + col("Away6s+RuckSpeed")) != 1).show()

#territory adds to 100%
df_bronze_match_data.filter(
    (col("TerritoryHomeTry-Line-22m") + col("TerritoryHome22m-50m") + col("TerritoryAway22m-50m") + col("TerritoryAwayTry-Line-22m") != 1)
).show()

#field pocession adds to 100%
df_bronze_match_data.filter((col("HomePossessionsTryLine-22m") + col("HomePossession22m-50m") + col("HomePossessionOpp50m-22m") + col("HomePossessionOpp22m-TryLine")) != 1).show()
df_bronze_match_data.filter((col("AwayPossessionsTryLine-22m") + col("AwayPossession22m-50m") + col("AwayPossessionOpp50m-22m") + col("AwayPossessionOpp22m-TryLine")) != 1).show()

#pocession in the last 10 adds to 100%
df_bronze_match_data.filter(
    (col("HomePossessionLast10") + col("AwayPossessionLast10")) != 1
).show()


In [0]:
#add meta data and store in the silver layer
#rename bronze meta data columns 
df_bronze_match_data = df_bronze_match_data.withColumnRenamed("ingestTimestamp", "ingestTimestampBronze").withColumnRenamed("sourceFile", "sourceFileBronze")
#add silver layer meta data 
df_bronze_match_data = df_bronze_match_data.withColumn("lastUpload", current_timestamp()).withColumn("pipelineStage", lit("silverTransformation"))

#write to silver table
df_bronze_match_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("rugby_data_dev.rugby_silver.match_results_data_")
